# `kitai.batch` — Guide

This notebook walks through every function in `kitai/batch.py` in the order
you would call them in a real workflow.

## Module overview

```
kitai/batch.py
│
├── Generic Batch API primitives          (any endpoint)
│   ├── submit_batch_job()                task list → job_id
│   ├── check_batch_job()                 job_id → status dict
│   ├── download_batch_results()          job_id → raw result list
│   ├── poll_until_complete()             blocks until all jobs finish
│   └── BatchJobNotCompleteError          raised when downloading too early
│
├── Embedding workflow helpers            (/v1/embeddings)
│   ├── build_embedding_tasks()           Documents → task dicts
│   └── parse_embedding_results()         raw results → (custom_id, embedding) pairs
│
└── Chat completion workflow helpers      (/v1/chat/completions)
    ├── build_chat_tasks()                items + system_prompt → task dicts
    └── parse_chat_results()              raw results + extractor_fn → (custom_id, T) pairs
```

## Sections
1. [Setup](#1-setup)
2. [Prepare documents](#2-prepare-documents)
3. [Embedding pipeline — step by step](#3-embedding-pipeline)
4. [Chat completions — high-level helpers](#4-chat-completions)
5. [Generic Batch API — manual tasks](#5-generic-batch-api)
6. [Error handling reference](#6-error-handling)

---
## 1 — Setup

Imports, logging, and the OpenAI client.

> **Prerequisite:** `OPENAI_API_KEY` must be set in your `.env` file or
> environment before running the cells below.

In [ ]:
import logging
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from langchain_core.documents import Document

from kitai.batch import (
    submit_batch_job,
    check_batch_job,
    download_batch_results,
    poll_until_complete,
    BatchJobNotCompleteError,
    build_embedding_tasks,
    parse_embedding_results,
    build_chat_tasks,
    parse_chat_results,
)

# Route kitai.batch logs to the notebook output.
# Switch to DEBUG to see every upload ID and poll tick.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    force=True,
)

load_dotenv()
client = OpenAI()

print("kitai.batch imported successfully")

---
## 2 — Prepare documents

`build_embedding_tasks` requires a list of LangChain `Document` objects where
every document carries a unique `metadata["id"]`.

`kitai.transform` provides helpers for building that list from different sources:

| Function | Responsibility |
|---|---|
| `list_to_docs(texts)` | `list[str]` → `list[Document]` |
| `df_to_docs(df, content_column, metadata_columns)` | `DataFrame` → `list[Document]` |
| `add_metadata_to_docs(docs, key, value)` | attach a constant metadata field to every doc |

After conversion, assign a unique integer `metadata["id"]` to each document —
this becomes the `custom_id` key in the Batch API tasks and lets you join
results back to your source data.

In [ ]:
from langchain_core.documents import Document
from kitai.transform import list_to_docs, df_to_docs

# --- Option A: build from a list of strings (quick demo) -----------------
raw_texts = [
    "Algorithmic trading involves using computer programs to execute trades.",
    "Short selling is a strategy where an investor borrows shares to sell them.",
    "Risk management is essential for any long/short equity strategy.",
]
docs = list_to_docs(raw_texts)

# Assign a unique metadata["id"] to each document (required by the batch pipeline)
for i, doc in enumerate(docs, start=1):
    doc.metadata["id"] = i

# --- Option B: build from a DataFrame ------------------------------------
# import pandas as pd
# df = pd.read_csv("./data/my_data.csv")
# docs = df_to_docs(df, content_column="description", metadata_columns=["source"])
# for i, doc in enumerate(docs, start=1):
#     doc.metadata["id"] = i

print(f"Documents loaded: {len(docs)}")
for doc in docs:
    print(f"  id={doc.metadata['id']}  text={doc.page_content[:60]}...")

---
## 3 — Embedding pipeline

The full flow in five steps:

```
docs
 └─► build_embedding_tasks()   — format for the Batch API
      └─► submit_batch_job()   — upload + create job → job_id
           └─► poll_until_complete()  — wait
                └─► download_batch_results()  — pull raw JSONL
                     └─► parse_embedding_results()  → [(custom_id, vector), ...]
```

### Step 1 — Build embedding tasks

`build_embedding_tasks` converts your documents into the dict schema the
OpenAI Batch API expects.  `custom_id` is set to the raw value of
`doc.metadata["id"]` — so you can join results back to your source data
using the same integer you assigned.

In [ ]:
tasks = build_embedding_tasks(
    docs,
    model="text-embedding-3-small",
)

print(f"Tasks built: {len(tasks)}")
print("\nFirst task:")
tasks[0]

### Step 2 — Submit the batch job

`submit_batch_job` serialises the task list to a temporary JSONL file,
uploads it to the OpenAI Files API, creates the batch job, and returns
the job ID.  The temp file is deleted regardless of success or failure.

In [ ]:
job_id = submit_batch_job(client, tasks)

print(f"Job ID: {job_id}")
# Save this — you can paste it into a later cell if the kernel restarts.
# job_id = "batch_abc123"

### Step 3 — Check status (manual spot-check)

Use `check_batch_job` to inspect the current state of a single job
without blocking.  Useful for debugging or when you want to poll manually.

Returns `job.model_dump()`.  Key fields:

| Key | Type | Meaning |
|---|---|---|
| `status` | str | e.g. `"in_progress"`, `"completed"`, `"failed"` |
| `request_counts` | dict | `{completed, failed, processing}` |
| `output_file_id` | str\|None | set when complete |
| `error_file_id` | str\|None | set when there are failed items |

Terminal statuses: `"completed"`, `"failed"`, `"expired"`, `"cancelled"`.

In [ ]:
status = check_batch_job(client, job_id)

counts = status.get("request_counts", {})
print(f"Status      : {status['status']}")
print(f"Completed   : {counts.get('completed', '?')}")
print(f"Failed      : {counts.get('failed', '?')}")
print(f"Output file : {status.get('output_file_id')}")

### Step 4 — Wait for completion

`poll_until_complete` blocks and polls every `poll_interval_seconds` seconds
until all jobs reach a terminal state.  It returns a `dict[job_id → status_dict]`
for **all** submitted jobs — including failed ones.  Check
`statuses[job_id]["status"] == "completed"` before downloading.

**Tips:**
- Use `poll_interval_seconds=5` for short jobs in development.
- For production batches use `30`–`60` to avoid hammering the API.
- You can pass multiple job IDs if you submitted more than one batch.
- Raises `TimeoutError` if `max_iterations` is exceeded (default 1000).

In [ ]:
statuses = poll_until_complete(
    client,
    [job_id],
    poll_interval_seconds=5,
)

print(f"Job status: {statuses[job_id]['status']}")

if statuses[job_id]["status"] != "completed":
    raise RuntimeError(
        f"Job ended with status '{statuses[job_id]['status']}' — "
        "check the OpenAI dashboard for details."
    )

### Step 5 — Download raw results

`download_batch_results` raises `BatchJobNotCompleteError` if the job is
not yet `"completed"`.  Once complete it downloads the output JSONL and
returns one dict per submitted task.

Each raw result dict looks like:
```json
{
    "custom_id": "1",
    "result": {
        "data": [ { "embedding": [...] } ]
    },
    "error": null
}
```
Items that failed at the API level will have a non-null `"error"` field.

In [ ]:
results = download_batch_results(client, job_id)

print(f"Result objects : {len(results)}")
print(f"Failed items   : {sum(1 for r in results if r.get('error'))}")
print("\nFirst raw result (embedding truncated):")
first = results[0]
print(f"  custom_id : {first['custom_id']}")
print(f"  error     : {first.get('error')}")
print(f"  embedding : {first['result']['data'][0]['embedding'][:4]} ...")

### Step 6 — Parse embeddings

`parse_embedding_results` extracts `(custom_id, embedding)` pairs from the
raw result list.  Items with errors are skipped and logged — the caller
receives only the successfully parsed pairs.

In [ ]:
pairs = parse_embedding_results(results)

print(f"Parsed pairs : {len(pairs)}")
for custom_id, embedding in pairs:
    print(f"  {custom_id}  dim={len(embedding)}  first_values={embedding[:3]}")

### Step 7 — Save to CSV (optional)

Convert pairs to a DataFrame and persist.  `custom_id` is already the raw
integer `metadata["id"]` — no string parsing needed.

In [ ]:
df_embeddings = pd.DataFrame(pairs, columns=["id", "embedding"])

output_path = "./book_embeddings.csv"
df_embeddings.to_csv(output_path, index=False)

print(f"Saved {len(df_embeddings)} rows to {output_path}")
df_embeddings.head()

---
## 4 — Chat completions — high-level helpers

`build_chat_tasks` and `parse_chat_results` mirror the embedding helpers
for chat completions.  Each item needs `"id"` and `"content"` keys;
one `system_prompt` applies to all items.

```
items  →  build_chat_tasks()  →  submit_batch_job()
                                       │
                               poll_until_complete()
                                       │
                               download_batch_results()
                                       │
                               parse_chat_results(extractor_fn)  →  [(id, T), ...]
```

In [ ]:
topics = [
    {"id": "topic-1", "content": "Headlines: Fed holds rates steady; inflation at 2.1%"},
    {"id": "topic-2", "content": "Headlines: Nvidia Q2 earnings beat; data-center revenue up 80%"},
    {"id": "topic-3", "content": "Headlines: Oil drops to $70 on OPEC output increase"},
]

system_prompt = (
    "Give a short topic label of 3-5 words. "
    "Be specific. Return only the label, no explanation."
)

chat_tasks = build_chat_tasks(topics, system_prompt=system_prompt, model="gpt-4o-mini")

print(f"Built {len(chat_tasks)} chat tasks")
chat_tasks[0]

In [ ]:
chat_job_id = submit_batch_job(client, chat_tasks)
print(f"Chat batch job: {chat_job_id}")

In [ ]:
chat_statuses = poll_until_complete(client, [chat_job_id], poll_interval_seconds=5)

if chat_statuses[chat_job_id]["status"] != "completed":
    raise RuntimeError(f"Chat batch failed: {chat_statuses[chat_job_id]['status']}")

chat_results = download_batch_results(client, chat_job_id)

# parse_chat_results applies extractor_fn to the response string.
# Exceptions in extractor_fn are caught and the item is skipped (logged at WARNING).
labels = dict(parse_chat_results(chat_results, extractor_fn=str.strip))

print(f"Downloaded {len(chat_results)} result(s)\n")
for topic_id, label in labels.items():
    print(f"  [{topic_id}] {label}")

---
## 5 — Generic Batch API — manual tasks

The three primitives (`submit_batch_job`, `check_batch_job`,
`download_batch_results`) are endpoint-agnostic.  Use them when you need
per-task model, temperature, or multi-turn messages that
`build_chat_tasks` doesn't support.

Each task body must match the payload of the target endpoint.

In [ ]:
questions = [
    "What is the core idea behind short selling?",
    "What does 'alpha' mean in quantitative finance?",
    "Define 'drawdown' in one sentence.",
]

# Manual task dicts — full control over model, temperature, messages
manual_tasks = [
    {
        "custom_id": f"q_{i}",
        "body": {
            "model": "gpt-4o-mini",
            "messages": [{"role": "user", "content": q}],
            "max_tokens": 120,
        },
    }
    for i, q in enumerate(questions)
]

print(f"Built {len(manual_tasks)} manual tasks")
manual_tasks[0]

In [ ]:
manual_job_id = submit_batch_job(client, manual_tasks)
manual_statuses = poll_until_complete(client, [manual_job_id], poll_interval_seconds=5)

if manual_statuses[manual_job_id]["status"] != "completed":
    raise RuntimeError(f"Manual batch failed: {manual_statuses[manual_job_id]['status']}")

manual_results = download_batch_results(client, manual_job_id)

# Manual parsing — result path for chat completions
print(f"Downloaded {len(manual_results)} result(s)\n")
for item in manual_results:
    if item.get("error"):
        print(f"[{item['custom_id']}] ERROR: {item['error']}")
    else:
        answer = item["result"]["choices"][0]["message"]["content"]
        print(f"[{item['custom_id']}] {answer.strip()[:120]}")
        print()

---
## 6 — Error handling reference

### `BatchJobNotCompleteError`

Raised by `download_batch_results` when the job is not yet `"completed"`.
Check `.status` to decide whether to retry or abort.

```
status               meaning                     action
─────────────────    ─────────────────────────   ─────────────────────────
"validating"         OpenAI checking the file    poll again later
"in_progress"        running                     poll again later
"finalizing"         writing output file         poll again later
"failed"             hard failure                inspect error_file_id
"expired"            exceeded completion_window  resubmit
"cancelled"          user-cancelled              resubmit if needed
```

In [ ]:
# Simulate calling download before a job is done.
# Replace with a real in-progress job ID to test against live API.
fake_in_progress_id = "batch_notdoneyetXXX"

try:
    download_batch_results(client, fake_in_progress_id)
except BatchJobNotCompleteError as e:
    print(f"Caught BatchJobNotCompleteError")
    print(f"  .batch_id = {e.batch_id}")
    print(f"  .status   = {e.status}")
    if e.status in ("in_progress", "finalizing", "validating"):
        print("  → Job is still running. Call poll_until_complete() to wait.")
    else:
        print(f"  → Terminal failure ({e.status}). Check the OpenAI dashboard.")
except Exception as e:
    # Will hit openai.NotFoundError for a fake ID — expected in this demo.
    print(f"API error (expected for fake ID): {type(e).__name__}: {e}")

### `ValueError` — empty inputs

All functions that take a list guard against empty input and raise
`ValueError` immediately, before any network call is made.

In [ ]:
guards = [
    ("build_embedding_tasks([])",      lambda: build_embedding_tasks([])),
    ("submit_batch_job(client, [])",   lambda: submit_batch_job(client, [])),
    ("poll_until_complete(client, [])",lambda: poll_until_complete(client, [])),
    ("parse_embedding_results([])",    lambda: parse_embedding_results([])),
]

for label, fn in guards:
    try:
        fn()
    except ValueError as e:
        print(f"{label}\n  → ValueError: {e}\n")

### Debugging tips

| Symptom | Where to look |
|---|---|
| Job never leaves `"validating"` | OpenAI is validating the uploaded file; wait a minute then re-check. |
| `status == "failed"` with `error_file_id` | Download the error file: `client.files.content(status['error_file_id']).text` |
| `parse_embedding_results` skips items | Set `logging.DEBUG` to see the exact `custom_id` and error for every skipped item. |
| `parse_chat_results` skips items | Same — extractor_fn exceptions are caught and logged at WARNING. |
| Need to re-run download after kernel restart | Copy the job ID from the log output and set `job_id = "batch_xxx"` manually. |
| `TimeoutError` from `poll_until_complete` | Increase `max_iterations` (default 1000) or re-poll manually after the job finishes. |